# 🏠 Tanzania Real Estate Data Exploration

This notebook contains comprehensive data analysis and exploration for the Tanzania Real Estate AI project.

## 📋 Table of Contents
1. Data Loading & Overview
2. Data Cleaning & Preprocessing
3. Exploratory Data Analysis
4. Feature Engineering
5. Visualization & Insights
6. Market Trends Analysis
7. Geographic Analysis
8. Price Prediction Modeling
9. Recommendations & Insights

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from datetime import datetime
import folium
from folium.plugins import HeatMap

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
plt.rcParams['figure.figsize'] = (12, 8)

print("📊 Libraries imported successfully!")
print(f"🕒 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Data Loading & Overview

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('../data/sample_data.csv')
    print("✅ Dataset loaded successfully!")
    print(f"📊 Dataset shape: {df.shape}")
    print(f"📅 Date range: {df['listing_date'].min()} to {df['listing_date'].max()}")
except FileNotFoundError:
    print("❌ Dataset not found. Please ensure the data file exists.")
    # Create a sample dataset for demonstration
    df = pd.DataFrame({
        'id': range(1, 31),
        'location': ['Kinondoni', 'Ilala', 'Ubungo', 'Temeke', 'Masaki'] * 6,
        'city': ['Dar es Salaam'] * 25 + ['Arusha'] * 3 + ['Dodoma'] * 2,
        'ward': ['Mikocheni', 'Kariakoo', 'Ubungo', 'Temeke', 'Masaki'] * 6,
        'price_tzs': np.random.randint(100000000, 1000000000, 30),
        'bedrooms': np.random.randint(1, 6, 30),
        'bathrooms': np.random.randint(1, 4, 30),
        'size_sqm': np.random.randint(50, 400, 30),
        'property_type': np.random.choice(['House', 'Apartment', 'Villa'], 30),
        'amenities': ['parking,security', 'security', 'parking,garden', 'security,swimming_pool', 'parking,security,gym'] * 6,
        'description': ['Modern property in prime location'] * 30,
        'listing_date': pd.date_range('2023-10-01', periods=30, freq='D'),
        'latitude': np.random.uniform(-7.0, -3.0, 30),
        'longitude': np.random.uniform(30.0, 40.0, 30)
    })
    print("📝 Created sample dataset for demonstration")

# Display first few rows
print("\n📋 First 5 rows:")
df.head()

In [ ]:
# Basic information about the dataset
print("📊 Dataset Information:")
df.info()

print("\n📈 Statistical Summary:")
df.describe().T

In [ ]:
# Check for missing values
print("❌ Missing Values:")
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Count': missing_values,
    'Percentage': missing_percentage
})

missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

## 2. Data Cleaning & Preprocessing

In [ ]:
# Convert date column to datetime
df['listing_date'] = pd.to_datetime(df['listing_date'])

# Extract additional features from date
df['listing_year'] = df['listing_date'].dt.year
df['listing_month'] = df['listing_date'].dt.month
df['listing_day_of_week'] = df['listing_date'].dt.dayofweek
df['days_since_listing'] = (datetime.now() - df['listing_date']).dt.days

# Calculate derived features
df['price_per_sqm'] = df['price_tzs'] / df['size_sqm']
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['size_per_room'] = df['size_sqm'] / df['total_rooms']

# Process amenities
df['amenities_count'] = df['amenities'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)

# Create binary features for specific amenities
amenity_features = ['parking', 'security', 'swimming_pool', 'garden', 'gym']
for feature in amenity_features:
    df[f'has_{feature}'] = df['amenities'].str.contains(feature, case=False, na=False).astype(int)

print("✅ Feature engineering completed!")
print(f"📊 New features added: {len(df.columns) - 15} additional columns")

In [ ]:
# Handle outliers in price
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    print(f"📊 {column}: Found {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)")
    
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Remove outliers from price and size
df_clean = remove_outliers_iqr(df, 'price_tzs')
df_clean = remove_outliers_iqr(df_clean, 'size_sqm')

print(f"\n📊 Original dataset: {len(df)} properties")
print(f"📊 After removing outliers: {len(df_clean)} properties")
print(f"📉 Removed {len(df) - len(df_clean)} properties ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")

## 3. Exploratory Data Analysis

In [ ]:
# Price distribution analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Price distribution
sns.histplot(df_clean['price_tzs'], bins=30, kde=True, ax=axes[0,0])
axes[0,0].set_title('Price Distribution (TZS)')
axes[0,0].set_xlabel('Price (TZS)')

# Price per sqm distribution
sns.histplot(df_clean['price_per_sqm'], bins=30, kde=True, ax=axes[0,1])
axes[0,1].set_title('Price per Square Meter Distribution')
axes[0,1].set_xlabel('Price per sqm (TZS)')

# Size distribution
sns.histplot(df_clean['size_sqm'], bins=30, kde=True, ax=axes[1,0])
axes[1,0].set_title('Property Size Distribution')
axes[1,0].set_xlabel('Size (sqm)')

# Bedrooms distribution
sns.countplot(data=df_clean, x='bedrooms', ax=axes[1,1])
axes[1,1].set_title('Bedrooms Distribution')
axes[1,1].set_xlabel('Number of Bedrooms')

plt.tight_layout()
plt.savefig('../outputs/charts/distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("📊 Distribution analysis completed and saved!")

In [ ]:
# Price by city analysis
plt.figure(figsize=(15, 6))

# Box plot of prices by city
plt.subplot(1, 2, 1)
sns.boxplot(data=df_clean, x='city', y='price_tzs')
plt.title('Price Distribution by City')
plt.ylabel('Price (TZS)')
plt.xticks(rotation=45)

# Average price by city
plt.subplot(1, 2, 2)
city_avg_prices = df_clean.groupby('city')['price_tzs'].mean().sort_values(ascending=False)
city_avg_prices.plot(kind='bar')
plt.title('Average Price by City')
plt.ylabel('Average Price (TZS)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('../outputs/charts/price_by_city.png', dpi=300, bbox_inches='tight')
plt.show()

print("📊 City-wise price analysis completed!")
print("\n🏙️ Average prices by city:")
for city, price in city_avg_prices.items():
    print(f"  {city}: {price:,.0f} TZS")

In [ ]:
# Property type analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Count by property type
property_counts = df_clean['property_type'].value_counts()
axes[0,0].pie(property_counts.values, labels=property_counts.index, autopct='%1.1f%%')
axes[0,0].set_title('Property Types Distribution')

# Average price by property type
type_prices = df_clean.groupby('property_type')['price_tzs'].mean()
axes[0,1].bar(type_prices.index, type_prices.values)
axes[0,1].set_title('Average Price by Property Type')
axes[0,1].set_ylabel('Average Price (TZS)')

# Price vs Size scatter plot
sns.scatterplot(data=df_clean, x='size_sqm', y='price_tzs', hue='property_type', ax=axes[1,0])
axes[1,0].set_title('Price vs Size by Property Type')
axes[1,0].set_xlabel('Size (sqm)')
axes[1,0].set_ylabel('Price (TZS)')

# Bedrooms vs Price
sns.boxplot(data=df_clean, x='bedrooms', y='price_tzs', ax=axes[1,1])
axes[1,1].set_title('Price by Number of Bedrooms')
axes[1,1].set_ylabel('Price (TZS)')

plt.tight_layout()
plt.savefig('../outputs/charts/property_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("📊 Property type analysis completed!")

## 4. Geographic Analysis

In [ ]:
# Create a map of property locations
# Calculate center point for the map
center_lat = df_clean['latitude'].mean()
center_lon = df_clean['longitude'].mean()

# Create base map
m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

# Add markers for each property
for idx, row in df_clean.iterrows():
    popup_text = f"""
    <b>{row['location']}</b><br>
    City: {row['city']}<br>
    Price: {row['price_tzs']:,.0f} TZS<br>
    Size: {row['size_sqm']} sqm<br>
    Bedrooms: {row['bedrooms']}<br>
    Type: {row['property_type']}
    """
    
    # Color based on price range
    if row['price_tzs'] < 200000000:
        color = 'green'
    elif row['price_tzs'] < 500000000:
        color = 'orange'
    else:
        color = 'red'
    
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=popup_text,
        tooltip=f"{row['location']} - {row['price_tzs']:,.0f} TZS",
        icon=folium.Icon(color=color, icon='home')
    ).add_to(m)

# Save the map
m.save('../outputs/charts/property_map.html')

print("🗺️ Interactive property map created!")
print("📍 Map saved as 'property_map.html' in outputs/charts/")

# Display the map (if in Jupyter)
display(m)

## 5. Correlation Analysis

In [ ]:
# Correlation matrix for numerical features
numerical_features = ['price_tzs', 'size_sqm', 'bedrooms', 'bathrooms', 
                     'price_per_sqm', 'total_rooms', 'size_per_room', 'amenities_count']

correlation_matrix = df_clean[numerical_features].corr()

# Create heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of Property Features')
plt.tight_layout()
plt.savefig('../outputs/charts/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("📊 Correlation analysis completed!")
print("\n🔗 Top correlations with price:")
price_correlations = correlation_matrix['price_tzs'].sort_values(ascending=False)
for feature, corr in price_correlations.items():
    if feature != 'price_tzs':
        print(f"  {feature}: {corr:.3f}")

## 6. Market Trends Analysis

In [ ]:
# Time series analysis of listings
df_clean['listing_month_name'] = df_clean['listing_date'].dt.strftime('%B')

# Listings by month
monthly_listings = df_clean.groupby('listing_month_name').size().reindex(
    ['January', 'February', 'March', 'April', 'May', 'June', 
     'July', 'August', 'September', 'October', 'November', 'December']
)

# Average price by month
monthly_avg_price = df_clean.groupby('listing_month_name')['price_tzs'].mean().reindex(
    ['January', 'February', 'March', 'April', 'May', 'June', 
     'July', 'August', 'September', 'October', 'November', 'December']
)

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Monthly listings trend
monthly_listings.plot(kind='line', marker='o', ax=axes[0])
axes[0].set_title('Monthly Property Listings Trend')
axes[0].set_ylabel('Number of Listings')
axes[0].grid(True, alpha=0.3)

# Monthly average price trend
monthly_avg_price.plot(kind='line', marker='s', color='red', ax=axes[1])
axes[1].set_title('Monthly Average Price Trend')
axes[1].set_ylabel('Average Price (TZS)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/charts/market_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("📈 Market trends analysis completed!")

## 7. Feature Engineering for ML

In [ ]:
# Create additional ML features
df_ml = df_clean.copy()

# Price categories
df_ml['price_category'] = pd.cut(df_ml['price_tzs'], 
                                 bins=[0, 200000000, 500000000, float('inf')],
                                 labels=['Budget', 'Mid-range', 'Premium'])

# Size categories
df_ml['size_category'] = pd.cut(df_ml['size_sqm'],
                                bins=[0, 100, 200, float('inf')],
                                labels=['Small', 'Medium', 'Large'])

# Room density (rooms per sqm)
df_ml['room_density'] = df_ml['total_rooms'] / df_ml['size_sqm']

# Price per bedroom
df_ml['price_per_bedroom'] = df_ml['price_tzs'] / df_ml['bedrooms']

# Bathroom to bedroom ratio
df_ml['bath_bed_ratio'] = df_ml['bathrooms'] / df_ml['bedrooms']

# Amenities score (weighted)
amenity_weights = {
    'has_parking': 0.2,
    'has_security': 0.3,
    'has_swimming_pool': 0.25,
    'has_garden': 0.15,
    'has_gym': 0.1
}

df_ml['amenities_score'] = sum(df_ml[feature] * weight 
                              for feature, weight in amenity_weights.items())

print("🔧 Feature engineering for ML completed!")
print(f"📊 Total features for ML: {len(df_ml.columns)}")
print("\n✨ New ML features created:")
new_features = ['price_category', 'size_category', 'room_density', 
               'price_per_bedroom', 'bath_bed_ratio', 'amenities_score']
for feature in new_features:
    print(f"  • {feature}")

## 8. Key Insights & Recommendations

In [ ]:
# Generate key insights
print("🎯 KEY MARKET INSIGHTS")
print("=" * 50)

# 1. Market Overview
print("\n📊 MARKET OVERVIEW:")
print(f"  • Total Properties Analyzed: {len(df_clean):,}")
print(f"  • Average Property Price: {df_clean['price_tzs'].mean():,.0f} TZS")
print(f"  • Median Property Price: {df_clean['price_tzs'].median():,.0f} TZS")
print(f"  • Average Size: {df_clean['size_sqm'].mean():.1f} sqm")
print(f"  • Price per sqm: {df_clean['price_per_sqm'].mean():,.0f} TZS")

# 2. City Analysis
print("\n🏙️ TOP CITIES BY AVERAGE PRICE:")
city_analysis = df_clean.groupby('city').agg({
    'price_tzs': ['mean', 'count'],
    'size_sqm': 'mean'
}).round(0)

city_analysis.columns = ['Avg Price (TZS)', 'Property Count', 'Avg Size (sqm)']
city_analysis = city_analysis.sort_values('Avg Price (TZS)', ascending=False)

for city, row in city_analysis.iterrows():
    print(f"  • {city}: {row['Avg Price (TZS)']:,.0f} TZS ({row['Property Count']} properties)")

# 3. Property Type Insights
print("\n🏠 PROPERTY TYPE INSIGHTS:")
type_analysis = df_clean.groupby('property_type').agg({
    'price_tzs': ['mean', 'median'],
    'size_sqm': 'mean',
    'bedrooms': 'mean'
}).round(1)

type_analysis.columns = ['Avg Price', 'Median Price', 'Avg Size', 'Avg Bedrooms']

for prop_type, row in type_analysis.iterrows():
    print(f"  • {prop_type}: {row['Avg Price']:,.0f} TZS avg, {row['Avg Size']:.0f} sqm avg")

# 4. Investment Recommendations
print("\n💡 INVESTMENT RECOMMENDATIONS:")
print("  🏆 BEST ROI AREAS:")
best_roi = df_clean[df_clean['price_per_sqm'] < df_clean['price_per_sqm'].quantile(0.6)]
best_cities = best_roi.groupby('city')['price_per_sqm'].mean().sort_values().head(3)

for city, price_per_sqm in best_cities.items():
    print(f"    • {city}: {price_per_sqm:,.0f} TZS per sqm (good value)")

print("\n  📈 HIGH DEMAND FEATURES:")
popular_amenities = df_clean[['has_parking', 'has_security', 'has_swimming_pool']].mean().sort_values(ascending=False)
for amenity, percentage in popular_amenities.items():
    feature_name = amenity.replace('has_', '').title()
    print(f"    • {feature_name}: {percentage*100:.1f}% of properties")

print("\n  🎯 TARGET SEGMENTS:")
print("    • Budget Properties: <200M TZS (good for first-time buyers)")
print("    • Mid-range Properties: 200M-500M TZS (balanced investment)")
print("    • Premium Properties: >500M TZS (high-end market)")

In [ ]:
# Save processed data for ML training
df_ml.to_csv('../data/cleaned/processed_properties.csv', index=False)
df_clean.to_csv('../data/cleaned/cleaned_properties.csv', index=False)

print("💾 Data saved for ML training:")
print("  • processed_properties.csv - with engineered features")
print("  • cleaned_properties.csv - cleaned basic data")

# Save analysis summary
summary = {
    'analysis_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_properties': len(df_clean),
    'average_price': float(df_clean['price_tzs'].mean()),
    'median_price': float(df_clean['price_tzs'].median()),
    'average_size': float(df_clean['size_sqm'].mean()),
    'cities_covered': df_clean['city'].nunique(),
    'property_types': df_clean['property_type'].unique().tolist(),
    'price_range': {
        'min': float(df_clean['price_tzs'].min()),
        'max': float(df_clean['price_tzs'].max()),
        'std': float(df_clean['price_tzs'].std())
    }
}

import json
with open('../outputs/charts/analysis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✅ Data exploration completed successfully!")
print(f"📊 Generated {len(df_clean.columns)} features for ML modeling")
print(f"📈 Created {len(os.listdir('../outputs/charts/'))} visualizations")
print(f"🕒 Analysis completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📋 Summary

This comprehensive data exploration notebook has:

✅ **Data Quality Check**: Loaded and cleaned the dataset, handling missing values and outliers

✅ **Feature Engineering**: Created 15+ new features for ML modeling

✅ **Market Analysis**: Analyzed prices by city, property type, and amenities

✅ **Geographic Visualization**: Created interactive property maps

✅ **Trend Analysis**: Identified market patterns and seasonality

✅ **Investment Insights**: Provided data-driven recommendations

### 🎯 Key Findings:
- Average property price: X TZS
- Top performing cities: [City names]
- Best ROI areas: [Locations]
- High-demand features: [Amenities]

### 📊 Next Steps:
1. Train ML models using engineered features
2. Implement web scraping for real-time data
3. Build API endpoints for predictions
4. Create interactive frontend dashboard

---
*Analysis completed for Tanzania Real Estate AI Project*